In [1]:
import sys
sys.path.append("..")

In [12]:
import numpy as np
import pandas as pd
import os
import plotly.express as px
import pickle

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

In [3]:
def calThetaAdv_l1(xP: np.ndarray, theta0: np.ndarray, alpha):
    thetaP = theta0.copy()
    i = np.argmax(np.abs(xP))
    thetaP[i] -= (alpha * np.sign(xP[i]))

    return thetaP

def calThetaAdv_linf(xP: np.ndarray, weights: np.ndarray, bias, alpha):
    # xP does not have bias
    weights_adv = weights - (alpha * np.sign(xP))

    for i in range(len(xP)):
        if np.sign(xP[i]) == 0:
            weights_adv[i] = weights_adv[i] - (alpha * np.sign(weights_adv[i]))
    bias_adv = bias - alpha

    return np.concat((weights_adv, [bias_adv]))

def calTheta(xP: np.array, theta0: np.array, alpha: float, methods: str):
    if "L1" in methods:
        return calThetaAdv_l1(xP, theta0, alpha)
    else:
        return calThetaAdv_linf(xP, theta0[:-1], theta0[-1], alpha)    

def getStats(xP: np.ndarray, x0: np.ndarray, theta: np.ndarray, lamb):
    if xP.size != theta.size:
        x0 = np.hstack((x0, 1))
        xP = np.hstack((xP, 1))

    return np.log(1 + np.exp(-(xP @ theta))) + \
        (lamb * (np.linalg.norm(x0 - xP, ord=1)))

In [ ]:
def readPickle(final_path: str):
    df = pd.read_pickle(final_path)
    df["theta_r"] = df.apply(lambda row : calTheta(row['x_r'], row['theta_0'], row['alpha'], row['algorithm']), axis=1)
    df['J'] = df.apply(lambda row: getStats(row['x_r'], row['x_0'], row['theta_r'], row['lambda']), axis=1)

    return df

def readPickleAll(dir_path: str, clf_name:str, dataset_name: str, lambda_list: list, alpha_list:list):
    all_files = []

    for file in os.listdir(dir_path):
        splitted_file = file.split(sep='_')
        if dataset_name in splitted_file and clf_name in splitted_file and float(splitted_file[3]) in lambda_list and float(splitted_file[4]) in alpha_list:
            all_files.append(os.path.join(dir_path, file))
            
    df = pd.concat([pd.read_pickle(f_n) for f_n in all_files])

    # df["theta_r"] = df.apply(lambda row : calTheta(row['x_r'], row['theta_0'], row['alpha'], row['algorithm']), axis=1)
    # df['J'] = df.apply(lambda row: getStats(row['x_r'], row['x_0'], row['theta_r'], row['lambda']), axis=1)
    # df = df.groupby("algorithm", group_keys=False).apply(lambda x : x.reset_index(drop=True))

    return df

In [111]:
def getDFFromPickle(data: dict, alg_name: str, lamb: float):  
    x0s = []
    try:
        theta0 = data['theta_0']['out.weight'][0].numpy().astype(np.float64)
        bias0 = data['theta_0']['out.bias'].numpy().astype(np.float64) 
    except TypeError:
        try: 
            theta0 = data['theta_0'][0][0][0][0].numpy().astype(np.float64)
            bias0 = data['theta_0'][0][0][1].numpy().astype(np.float64)
        except IndexError:
            theta0 = data['theta_0'][0][0][0].numpy().astype(np.float64)
            bias0 = data['theta_0'][0][1].numpy().astype(np.float64)
    theta0 = np.concat((theta0, bias0))
    xRs = []
    alphas = []
    theta0s = []
    lambdas = []
    alg_names = []

    for para_i, para in enumerate(data['params']):
        for j in range(len(data['x_0'][para_i])):
            x0s.append(data['x_0'][para_i][j])
            xRs.append(data['x_r'][para_i][j])
            alphas.append(para['delta_max'])
            theta0s.append(theta0)
            lambdas.append(lamb)
            alg_names.append(alg_name)
    
    df = pd.DataFrame({'alg': alg_names,'x_0': x0s, 'theta_0' : theta0s, 'x_R' : xRs, 'alpha': alphas, 'lambda': lambdas})
    return df

In [119]:
dir_path = "../results/cost_validity_latest"
clf_name = "lr"
dataset_name = "synthesis"
pos_alg_names = ["alg1", "L1PSD", "ROARL1", "ROARLInf"]
alg_name = ["alg1", "GS", "ROARLInf"]
lambdas = [0.2]

df_column_names = ["alg", "x_0", "theta_0", "x_R", "alpha", "lambda"]
df = pd.DataFrame(columns=df_column_names)

all_files = []
for file in os.listdir(dir_path):
    splitted_file = file.split(sep='_')
    if dataset_name in splitted_file and clf_name in splitted_file and splitted_file[-1] == "new.pickle":
        lamb = float(splitted_file[3].replace('lamb', ''))
        if lamb in lambdas and splitted_file[2] in alg_name:
            all_files.append(os.path.join(dir_path, file))

for file in all_files:
    with open(file, 'rb') as f:
        data = pickle.load(f)
        splitted_file = file.split(sep='_') 
        tmp_alg_name = splitted_file[4]
        lamb = float(splitted_file[5].replace('lamb', ''))
        df = pd.concat([df, getDFFromPickle(data, tmp_alg_name, lamb)], ignore_index=True)

df["theta_R"] = df.apply(lambda row : calTheta(row['x_R'], row['theta_0'], row['alpha'], row['alg']), axis=1)
df['J'] = df.apply(lambda row: getStats(row['x_R'], row['x_0'], row['theta_R'], row['lambda']), axis=1)

C:\Users\pmyat\AppData\Local\Temp\ipykernel_28420\4283201931.py:25: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



In [120]:
df_mean = df.groupby(['alg', 'lambda', 'alpha'], as_index=False).mean()
df_mean

,alg,lambda,alpha,x_0,theta_0,x_R,theta_R,J
0,GS,0.2,0.1,"[-0.3831824262078427, 2.5503849272755756]","[1.0254535675048828, -0.17008450627326965, -0....","[2.7211890000000016, 2.481271500000001]","[0.9254535675048838, -0.2200845062732697, -1.0...",1.111662
1,GS,0.2,0.2,"[-0.3831824262078427, 2.5503849272755756]","[1.0254535675048828, -0.17008450627326965, -0....","[3.1401890000000003, 1.8142715000000005]","[0.8254535675048813, -0.27008450627326946, -1....",1.323082
2,GS,0.2,0.3,"[-0.3831824262078427, 2.5503849272755756]","[1.0254535675048828, -0.17008450627326965, -0....","[3.552688999999999, 1.4027714999999998]","[0.7254535675048841, -0.2960845062732698, -1.2...",1.532008
3,GS,0.2,0.4,"[-0.3831824262078427, 2.5503849272755756]","[1.0254535675048828, -0.17008450627326965, -0....","[3.9376889999999998, 0.9687714999999997]","[0.6254535675048838, -0.28208450627326953, -1....",1.761210
4,GS,0.2,0.5,"[-0.3831824262078427, 2.5503849272755756]","[1.0254535675048828, -0.17008450627326965, -0....","[3.1081890000000003, -0.21272849999999996]","[0.5254535675048828, -0.21008450627326966, -1....",1.955054
5,GS,0.2,0.6,"[-0.3831824262078427, 2.5503849272755756]","[1.0254535675048828, -0.17008450627326965, -0....","[3.1391890000000005, 0.0012714999999999996]","[0.42545356750488206, -0.17008450627326965, -1...",2.164867
6,GS,0.2,0.7,"[-0.3831824262078427, 2.5503849272755756]","[1.0254535675048828, -0.17008450627326965, -0....","[2.9786890000000006, -0.0002285000000000002]","[0.32545356750488325, -0.1280845062732697, -1....",2.403267
7,GS,0.2,0.8,"[-0.3831824262078427, 2.5503849272755756]","[1.0254535675048828, -0.17008450627326965, -0....","[0.3176890000000001, -0.0037284999999999974]","[0.2414535675048824, -0.010084506273269676, -1...",2.623156
8,ROARLInf,0.2,0.1,"[-0.3831824262078427, 2.5503849272755756]","[1.0254535675048828, -0.17008450627326965, -0....","[1.856280517578125, 2.550694885253906]","[0.9254535660147667, -0.2200845070183277, -1.0...",1.190988
9,ROARLInf,0.2,0.2,"[-0.3831824262078427, 2.5503849272755756]","[1.0254535675048828, -0.17008450627326965, -0....","[3.032073974609375, 2.550642852783203]","[0.8254535645246506, -0.2700845077633858, -1.1...",1.246763


In [122]:
custom_colors = {
    "LInf": "#33FFFF",
    "L1PSD": "#FF3333",
    "ROARLInf": "#33FF33",
    "ROARL1": "#FF33FF",
    "GS": "#FF3333"
}


fig = px.line(df_mean, x = "alpha", y = "J", color="alg", 
           labels = {"seed" : "Fold", "J" : "Total Cost (J)"},
           title=f"{clf_name}_{dataset_name}_lambda{lamb}_costMean",
            markers=True,
            color_discrete_map=custom_colors )
fig

# fig.write_html(f"{clf_name}_{dataset_name}_lambda{lamb}_costMean.html")
# fig

In [ ]:
# rng = np.random.default_rng(seed=seed)
# size_N = int(np.rint(0.15 * recourse_needed_X_test.shape[0]))
# recourse_needed_X_test = rng.choice(recourse_needed_X_test, size=size_N, replace=False)

output.groupby(['lambda', 'alpha', 'algorithm', 'seed']).count().apply(lambda row: np.arange(0,row['i']),axis=1)

lambda  alpha  algorithm  seed
0.1     0.0    Alg1       0       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                          1       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                          2       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                          3       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                          4       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                                                        ...                        
0.3     0.5    ROARLInf   0       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                          1       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                          2       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                          3       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
                          4       [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...
Length: 200, dtype: object

In [70]:
# Comparsion between which feature changes

diff_value = 1e-3

output["cost"] = np.linalg.norm(np.stack(output["x_0"].values) - np.stack(output["x_r"].values), 1, 1)
output['diff'] = output.apply(lambda row : np.abs(row['x_r'] - row['x_0']), axis=1)
output['diff_bool'] = output['diff'].apply(lambda x: np.where(x > diff_value, 1, 0))
# output.sort_values(by=["seed"]).sort_index()

# output_with_i = output.copy()
# output_with_i["row_index"] = output.index
# output_diff = output_with_i.sort_values(["seed", "row_index", "algorithm"]).drop(columns="row_index")

output = output.reset_index(drop=True)
output['sparsity'] = output['diff_bool'].apply(lambda x: sum(x)).reset_index(drop=True)
# output[output['sparsity'] == 25]

print(np.stack(output['theta_0'].abs()).min(axis=1))


# output['cost'].describe()
# output_diff['diff_bool'].apply(lambda x: sum(x))
px.scatter(output, y="cost", x="sparsity", color="algorithm")


[0.0798 0.0798 0.0798 ... 0.0723 0.0723 0.0723]


In [123]:
dir_path = "../results/recourse"
file_name = "lr_synthetic_L1PSD_1.pkl"
final_path = os.path.join(dir_path, file_name)

# df2 = pd.read_pickle(final_path)
# # df2 = df2.rename(columns={"x_r": "theta_0", "theta_0": "x_r"})
# df2['theta_r'] = df2.apply(lambda row : calThetaAdv_l1(np.hstack((row['x_r'], 1)), row['theta_0'], row['alpha']), axis=1)
# df2['J'] = df2.apply(lambda row: getStats(row['x_r'], row['x_0'], row['theta_r'], row['lambda']), axis=1)
# # df2.loc[:,['algorithm', 'seed', 'alpha', 'lambda', 'i', 'x_0', 'x_r', 'theta_0']]

# # df2.to_pickle(final_path)

# df2

df2 = readPickle(final_path)

In [ ]:
dir_path = "../results/recourse"
file_name = "lr_synthetic_ROAR_0.pkl"
final_path = os.path.join(dir_path, file_name)

df3 = pd.read_pickle(final_path)